# 02_快速入门

**来源:**
- https://docs.langchain.com/docs/quickstart
- https://docs.langchain.com/docs/install

## 1. 安装 LangChain

### 基础安装

```bash
# 使用 pip
pip install -U langchain
# 需要 Python 3.10+

# 使用 uv
uv add langchain
```

### 安装提供商集成包

LangChain 提供数百个 LLM 和数千个其他集成的独立包：

```bash
# OpenAI 集成
pip install -U langchain-openai
# 或
uv add langchain-openai

# Anthropic 集成
pip install -U langchain-anthropic
uv add langchain-anthropic

# 其他提供商同理
```

### 快速入门依赖

```bash
# pip 方式
pip install -U langchain deepagents

# uv 方式
uv init
uv add langchain deepagents
uv sync

# venv 方式
python3 -m venv .venv
source .venv/bin/activate
pip install -U langchain deepagents
```

### 设置 API 密钥

```bash
# OpenAI
export OPENAI_API_KEY="your-api-key"

# Google Gemini
export GOOGLE_API_KEY="your-api-key"

# Anthropic Claude
export ANTHROPIC_API_KEY="your-api-key"

# OpenRouter
export OPENROUTER_API_KEY="your-api-key"

# Azure OpenAI
export AZURE_OPENAI_API_KEY="your-api-key"
export AZURE_OPENAI_ENDPOINT="https://your-resource.openai.azure.com"
export AZURE_OPENAI_DEPLOYMENT_NAME="your-deployment"

# AWS Bedrock
export AWS_ACCESS_KEY_ID="your-access-key"
export AWS_SECRET_ACCESS_KEY="your-secret-key"
export AWS_REGION="us-east-1"

# HuggingFace
export HUGGINGFACEHUB_API_TOKEN="hf_..."

# Ollama (本地运行)
# 确保 Ollama 在运行: https://ollama.com
```

## 2. 构建基础 Agent

以下示例创建一个简单的 LangChain Agent，带有一个自定义天气工具：

In [1]:
# 导入 create_agent 函数
from langchain.agents import create_agent
from dotenv import load_dotenv
load_dotenv()
# 定义一个工具函数，用于获取天气
# 函数名、参数名和文档字符串会成为模型可见的工具描述

def get_weather(city: str) -> str:
    """获取指定城市的天气"""
    return f"{city}的天气总是晴朗！"

# 创建 Agent
# model: 模型名称（格式：provider:model_name）
# tools: 可用的工具列表
# system_prompt: 系统提示词，定义 Agent 的行为

agent = create_agent(
    model="deepseek-v4-flash",  # 可切换为其他模型
    tools=[get_weather],
    system_prompt="你是一个乐于助人的助手",
)

# 调用 Agent，传入消息
result = agent.invoke(
    {"messages": [{"role": "user", "content": "旧金山的天气怎么样？"}]}
)
print(result["messages"][-1].content_blocks)

[{'type': 'reasoning', 'reasoning': '工具返回了结果：旧金山的天气总是晴朗。那我就如实回答用户。'}, {'type': 'text', 'text': '根据查询结果，旧金山的天气总是晴朗的！☀️ 阳光明媚，是个好天气。如果你有更多问题，随时问我哦！'}]


### 使用不同的模型提供商

你只需更改 `model` 参数即可切换模型：

```python
# Google Gemini
agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

# Anthropic Claude
agent = create_agent(
    model="claude-sonnet-4-6",
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

# Ollama 本地模型
agent = create_agent(
    model="ollama:devstral-2",
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

# 使用 Azure OpenAI
import os
agent = create_agent(
    model="azure_openai:gpt-5.4",
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
    azure_deployment=os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"],
)

# 使用 AWS Bedrock
agent = create_agent(
    model="anthropic.claude-3-5-sonnet-20240620-v1:0",
    model_provider="bedrock_converse",
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)
```

## 3. 构建真实的 Agent

下面构建一个能够回答文本文件问题的研究 Agent，涵盖以下概念：

- **详细的系统提示词**：更好的 Agent 行为
- **创建工具**：与外部数据集成
- **模型配置**：一致的响应
- **对话记忆**：聊天式交互
- **Deep Agents**：内置功能
- **测试 Agent**

### 3.1 定义系统提示词

系统提示词定义了 Agent 的角色和行为，需要具体且可操作：

In [2]:
SYSTEM_PROMPT = """你是一个文学数据助手。

## 能力

- `fetch_text_from_url`: 从 URL 加载文档文本到对话中。
不要猜测行数或位置——将它们基于保存文件中的工具结果。
"""

print(SYSTEM_PROMPT)

你是一个文学数据助手。

## 能力

- `fetch_text_from_url`: 从 URL 加载文档文本到对话中。
不要猜测行数或位置——将它们基于保存文件中的工具结果。



### 3.2 创建工具

工具让模型能够与外部系统交互。使用 `@tool` 装饰器：

In [3]:
import urllib.error
import urllib.request

from langchain.tools import tool


@tool
def fetch_text_from_url(url: str) -> str:
    """从指定 URL 获取文档内容"""
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0 (compatible; quickstart-research/1.0)"},
    )
    try:
        with urllib.request.urlopen(req, timeout=120) as resp:
            raw = resp.read()
    except urllib.error.URLError as e:
        return f"获取失败: {e}"
    text = raw.decode("utf-8", errors="replace")
    return text

print(fetch_text_from_url.name)  # 工具名称
print(fetch_text_from_url.args)   # 参数信息

fetch_text_from_url
{'url': {'title': 'Url', 'type': 'string'}}


工具的命名规范：
- 使用蛇形命名法（snake_case），例如 `web_search`
- 避免使用空格或特殊字符
- 文档字符串应简洁清晰，帮助模型理解工具的用途

### 3.3 配置模型参数

使用 `init_chat_model` 初始化模型并配置参数：

In [4]:
from langchain.chat_models import init_chat_model

# 配置模型参数
model = init_chat_model(
    "deepseek-v4-pro",
    temperature=0.5,      # 控制随机性，0-1之间
    timeout=300,          # 超时时间（秒）
    max_tokens=25000,     # 最大输出 token 数
)

print(f"模型: {model}")
print(f"温度: {model.temperature}")
print(f"最大 token: {model.max_tokens}")

模型: output_version=None client=<openai.resources.chat.completions.completions.Completions object at 0x000002AEA576AC10> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002AEA576B390> root_client=<openai.OpenAI object at 0x000002AEA57691D0> root_async_client=<openai.AsyncOpenAI object at 0x000002AEA576AD50> model_name='deepseek-v4-pro' temperature=0.5 model_kwargs={} openai_api_key=None openai_proxy=None request_timeout=300.0 max_tokens=25000 stream_chunk_timeout=120.0 api_key=SecretStr('**********') api_base='https://api.deepseek.com/v1'
温度: 0.5
最大 token: 25000


### 3.4 添加记忆（Checkpointer）

通过 `InMemorySaver` 为 Agent 添加短期记忆：

In [5]:
from langgraph.checkpoint.memory import InMemorySaver

# 创建内存检查点（记忆）
checkpointer = InMemorySaver()

# 创建带记忆的 Agent
research_agent = create_agent(
    model="deepseek-v4-pro",
    tools=[fetch_text_from_url],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
)

print("Agent 已创建，带有短期记忆功能")
print("提示：使用 thread_id 来区分不同对话")
print('config = {"configurable": {"thread_id": "会话-1"}}')

Agent 已创建，带有短期记忆功能
提示：使用 thread_id 来区分不同对话
config = {"configurable": {"thread_id": "会话-1"}}


### 集成 LangSmith 调试

设置环境变量以启用 LangSmith 追踪：

```bash
export LANGSMITH_TRACING="true"
export LANGSMITH_API_KEY="your-api-key"
```

LangSmith 可以跟踪 Agent 的每一步操作，包括：
- 模型调用
- 工具执行
- 中间状态
- 最终响应

---

**参考链接**
- https://docs.langchain.com/docs/quickstart
- https://docs.langchain.com/docs/install
- https://docs.langchain.com/docs/tracing/quickstart